# SOXL HOLD 模拟试运行入口

用户明确授权“仅在模拟盘试运行新候选，保留未晋升标记”。本卡只读验收已有SIM release、QLib行情和恢复后的原账本，不下单、不修改active配置；执行仍由既有scheduler承担。研究仍未晋升，原最大回撤和CVaR门未过。账户沿用2026-08-03开始的周期，不重置。14.94%仅指continuous-weight历史回放的开盘调仓日减少，不保证真实填单次数减少同一幅度。


In [1]:
from pathlib import Path
import os, sys, json, hashlib, tempfile
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE', 'true')
root = Path.cwd().resolve()
while not (root / 'AGENTS.md').exists():
    if root == root.parent: raise RuntimeError('workspace not found')
    root = root.parent
sys.path.insert(0, str(root))
import qlib
from qlib.data import D
from qlib.constant import REG_US
from qlib.workflow import R
from src.workflow.stage1_runtime_release import load_runtime_release
from src.workflow.stage1_futu_sim import build_runtime_target, runtime_state_from_paper_ledger
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.trainable_regime_policy_nested_wf import _remove_qlib_code_snapshot_artifacts
card = root / 'production/stage1_hold_paper_trial_20260928'
def digest(path): return hashlib.sha256(path.read_bytes()).hexdigest()


2026/09/28 17:11:57 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
release_path = root / 'production/stage1_futu_sim_v1/runtime_release.json'
cycle_path = root / 'runtime/soxl/stage1_futu_sim/paper_cycle.json'
cycle = json.loads(cycle_path.read_text())
ledger_path = cycle_path.parent / cycle['ledger_file']
ledger = json.loads(ledger_path.read_text())
input_hashes = {str(p.relative_to(root)): digest(p) for p in (release_path, cycle_path, ledger_path)}
loaded = load_runtime_release(root, release_path)
release = loaded['release']
assert release['execution_environment'] == 'SIMULATE'
assert release['research_promotion_status'] == 'not_promoted'
assert cycle['cycle_id'] == ledger['cycle_id'] == 'stage1_2026-08-03_v2'
assert ledger['inception_date'] == '2026-08-03'
assert loaded['paper_trial_authorization'] is not None


In [3]:
# All QLib initialization, D API reads and inference precede R.start.
provider = Path(cycle['marks_provider_uri'])
if not provider.is_absolute(): provider = root / provider
qlib.init(provider_uri=str(provider), region=REG_US, kernels=1)
calendar = D.calendar(start_time='2026-08-03', end_time='2026-09-28')
last_settled = str(calendar[-1])[:10]
close_marks = D.features(['SOXL','SOXX'], ['$close'], start_time=last_settled, end_time=last_settled)
assert len(close_marks) == 2 and close_marks['$close'].notna().all()
state = runtime_state_from_paper_ledger(ledger, recovery_spec=IntradayRecoveryDirectAllocationSpec(root=root))
target = build_runtime_target(root=root, current_prices=None, live_controller_state=state)
assert target['status'] == 'needs_execution_mark', target
assert input_hashes == {str(p.relative_to(root)): digest(p) for p in (release_path, cycle_path, ledger_path)}
evidence = {'release_id': release['release_id'], 'release_sha256': digest(release_path), 'input_hashes': input_hashes, 'execution_environment': 'SIMULATE', 'research_promotion_status': 'not_promoted', 'paper_trial_authorization_verified': True, 'cycle_id': ledger['cycle_id'], 'inception_date': ledger['inception_date'], 'initial_equity': ledger['initial_equity'], 'confirmed_fill_count': len(ledger['fills']), 'positions': ledger['positions'], 'ledger_equity': ledger['equity'], 'ledger_cash': ledger['cash'], 'provider_uri': str(provider), 'qlib_api': ['qlib.init', 'D.calendar', 'D.features'], 'latest_settled_session': last_settled, 'signal_date': target['signal_date'], 'target_status_without_execution_mark': target['status'], 'active_configuration_changed': False, 'orders_submitted': False, 'allocation_recorder_id': release['allocation']['recorder_id'], 'action_recorder_id': release['action']['recorder_id'], 'research_development_gates': loaded['action_summary']['development_gates'], 'authorized_failed_gates': [key for key, passed in loaded['action_summary']['development_gates'].items() if not passed], 'backtest_open_trade_day_reduction_pct': loaded['action_summary']['frequency_change']['open_trade_day_reduction_pct'], 'frequency_scope': 'continuous_weight_backtest_open_trade_days_not_actual_fill_count_guarantee'}
evidence


[61976:MainThread](2026-09-28 17:11:58,806) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61976:MainThread](2026-09-28 17:11:58,808) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61976:MainThread](2026-09-28 17:11:58,808) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_account_marks')}


[61976:MainThread](2026-09-28 17:11:58,815) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61976:MainThread](2026-09-28 17:11:58,815) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


...

[61976:MainThread](2026-09-28 17:11:59,072) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/dynamic_soxx_soxl_cash_allocation_wf_v1')}


[61976:MainThread](2026-09-28 17:11:59,167) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61976:MainThread](2026-09-28 17:11:59,168) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61976:MainThread](2026-09-28 17:11:59,168) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


{'release_id': 'stage1_hold_net_fees_20260928_paper_trial_v1',
 'release_sha256': '8e337aa5273508e13eca2d75e446dcd984bc12515743798416c287b4c6b36279',
 'input_hashes': {'production/stage1_futu_sim_v1/runtime_release.json': '8e337aa5273508e13eca2d75e446dcd984bc12515743798416c287b4c6b36279',
  'runtime/soxl/stage1_futu_sim/paper_cycle.json': 'aedefc1acf16ed0696ed012d9918f3012e5a83d0007e493aa9eaf625bd4faa3d',
  'runtime/soxl/stage1_futu_sim/paper_ledger_v2.json': '66fccb5e85216c7f28f7fbd80850a44c06e8a5e2b0266bcde6bb5de69e2a386e'},
 'execution_environment': 'SIMULATE',
 'research_promotion_status': 'not_promoted',
 'paper_trial_authorization_verified': True,
 'cycle_id': 'stage1_2026-08-03_v2',
 'inception_date': '2026-08-03',
 'initial_equity': 100000.0,
 'confirmed_fill_count': 20,
 'positions': {'SOXL': 200, 'SOXX': 98},
 'ledger_equity': 116320.32900000001,
 'ledger_cash': 32959.769,
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/stage1_account_marks',
 'qlib_api': ['qlib.init

In [4]:
R.set_uri(str(card / 'qlib_recorder'))
with tempfile.TemporaryDirectory(prefix='stage1_paper_preflight_') as tmp:
    path = Path(tmp) / 'preflight_evidence.json'
    path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2) + '\n')
    with R.start(experiment_name='stage1_hold_paper_trial_20260928'):
        recorder = R.get_recorder()
        R.log_params(release_id=release['release_id'], execution_environment='SIMULATE', research_promotion_status='not_promoted', ledger_sha256=digest(ledger_path))
        R.log_metrics(confirmed_fill_count=float(len(ledger['fills'])), account_reset=0.0, submitted_orders=0.0)
        recorder.log_artifact(str(path), artifact_path='preflight')
        recorder_info = {'recorder_id': recorder.id, 'recorder_local_dir': recorder.get_local_dir()}
_remove_qlib_code_snapshot_artifacts(Path(recorder_info['recorder_local_dir']))
summary = {**evidence, **recorder_info, 'experiment': card.name, 'card_type': 'authorized_unpromoted_simulate_trial', 'decision': 'user_authorized_simulate_only_trial_not_research_promotion', 'development_gate_pass': False, 'preflight_status': 'passed_read_only_waiting_for_real_execution_mark', 'artifact_policy': {'policy': 'slim_card_recorder_first', 'workflow_config': 'workflow.yaml', 'recorder_only_artifacts': ['artifacts/preflight/preflight_evidence.json']}, 'qlib_docs_reference': ['https://qlib.readthedocs.io/en/latest/component/workflow.html', 'https://qlib.readthedocs.io/en/latest/component/data.html']}
(card / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n')
(card / 'workflow.yaml').write_text('experiment: stage1_hold_paper_trial_20260928\nexecution_environment: SIMULATE\nresearch_promotion_status: not_promoted\nrelease_path: production/stage1_futu_sim_v1/runtime_release.json\naccount_cycle: stage1_2026-08-03_v2\naccount_reset: false\nnotebook_action: read_only_preflight\nruntime_entrypoint: production/run_stage1_futu_sim.sh\nprovider_uri: ' + str(provider) + '\nrecorder_id: ' + recorder.id + '\n')
conclusion = f'''# 中文结论

- 用户已明确授权仅在富途SIMULATE试运行新候选，研究状态仍为未晋升。最大回撤/CVaR两项失败仅按精确授权收据允许模拟观察；其余身份、时序和账本硬门保留。
- release及模型SHA校验通过。账户沿用2026-08-03周期，已恢复{len(ledger['fills'])}笔真实模拟成交；本notebook未重置账本、未修改active配置、未提交委托。
- QLib D API最新结算日为{last_settled}。不提供未来或伪造执行价时，目标生成正确返回`needs_execution_mark`，等待实际开盘价格。
- 历史连续权重回放开盘调仓日减少{summary['backtest_open_trade_day_reduction_pct']:.2f}%，该比例不保证实际委托/成交笔数。真实效果须从该账户后续成交验证。
- 实际运行沿用`production/run_stage1_futu_sim.sh`及既有scheduler；本卡是只读验收和证据入口。
- QLib recorder：{recorder.id}。
'''
(card / 'conclusion.md').write_text(conclusion)
print(conclusion)


[61976:MainThread](2026-09-28 17:11:59,210) INFO - qlib.workflow - [exp.py:258] - Experiment 204932422184330359 starts running ...


[61976:MainThread](2026-09-28 17:11:59,282) INFO - qlib.workflow - [recorder.py:345] - Recorder 1b62099d14d84e698e8a2f7524b46482 starts running under Experiment 204932422184330359 ...


[61976:MainThread](2026-09-28 17:11:59,768) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


# 中文结论

- 用户已明确授权仅在富途SIMULATE试运行新候选，研究状态仍为未晋升。最大回撤/CVaR两项失败仅按精确授权收据允许模拟观察；其余身份、时序和账本硬门保留。
- release及模型SHA校验通过。账户沿用2026-08-03周期，已恢复20笔真实模拟成交；本notebook未重置账本、未修改active配置、未提交委托。
- QLib D API最新结算日为2026-09-25。不提供未来或伪造执行价时，目标生成正确返回`needs_execution_mark`，等待实际开盘价格。
- 历史连续权重回放开盘调仓日减少14.94%，该比例不保证实际委托/成交笔数。真实效果须从该账户后续成交验证。
- 实际运行沿用`production/run_stage1_futu_sim.sh`及既有scheduler；本卡是只读验收和证据入口。
- QLib recorder：1b62099d14d84e698e8a2f7524b46482。



## 后续观察

跟踪真实HOLD次数、开盘交易日、成交笔数、换手、滑点及回撤；模拟观察不自动改变研究晋升状态，也不授权REAL交易。
